# HOI Pose Estimation - Main Notebook

**PoliTo Advanced ML 2026-27** - Hand-Object Interaction with Transformers (HaMeR) on DexYCB

Single notebook for the whole workflow:
1. **Setup** - Clone the repo, install deps, mount *your* Drive, prepare the data, W&B login
2. **Data Exploration** - Inspect DexYCB, verify active object logic (Part 1)
3. **HaMeR Inference** - Pretrained evaluation (Table 1)
4. **HaMeR Fine-tuning** - Train on DexYCB (Table 1)
5. **HOI Training** - Full hand-object model (Part 3)
6. **Evaluation** - Table 2 metrics + visualizations (Figure 4)

**How we work**
- **Code** -> GitHub. Edit the `.py` files locally, `git push`, then re-run cell 1.1 here (`git pull`).
- **Data, MANO, checkpoints, outputs** -> your own Google Drive, folder `MyDrive/HOI_Project` (created by this notebook).
- **Metrics and curves** -> Weights & Biases.

**Before the first run**
- Runtime -> Change runtime type -> **T4 GPU**
- Colab Secrets (key icon on the left): add `WANDB_API_KEY` (from https://wandb.ai/authorize) and enable notebook access.
- Upload `MANO_RIGHT.pkl` and `MANO_LEFT.pkl` into `MyDrive/HOI_Project/mano/` (register at https://mano.is.tue.mpg.de).

Sections 2-6 need the parts of `src/` that are still `NotImplementedError`: run them once each part is implemented.

---
## 1. SETUP

In [ ]:
# 1.1 Clone the repository (first time) or update it (git pull)
import os

REPO_URL = "https://github.com/mikiungolo/Estimating-Poses-of-Hands-Interacting-with-Objects.git"
REPO_PATH = "/content/Estimating-Poses-of-Hands-Interacting-with-Objects"
BRANCH = "main"  # set your own branch to test it on Colab

if not os.path.exists(REPO_PATH):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_PATH}
else:
    !git -C {REPO_PATH} fetch -q && git -C {REPO_PATH} checkout -q {BRANCH} && git -C {REPO_PATH} pull -q

%cd {REPO_PATH}
!git log -1 --oneline
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# 1.2 Install the dependencies listed in pyproject.toml into Colab's Python
# (uv pip --system: the Colab kernel does not see a separate .venv)
!pip install -q uv
!uv pip install --system -q -r pyproject.toml
print("Dependencies installed")

In [ ]:
# 1.3 Mount YOUR Google Drive and create the project folders
import sys

from google.colab import drive

drive.mount("/content/drive")

sys.path.insert(0, REPO_PATH)
from src.utils.config import load_config

config = load_config("configs/config.yaml")

for d in [config.drive_dataset_dir, config.mano_dir, config.checkpoint_dir,
          config.log_dir, config.output_dir]:
    os.makedirs(d, exist_ok=True)

print(f"Your project folder on Drive: {config.drive_root}")
!ls {config.drive_root}

In [ ]:
# 1.4 Prepare the data
#  - first run ever: downloads DexYCB subject-01 + calibration + YCB models (~13.9 GB) into YOUR Drive
#  - every session: copies the archives to the Colab disk and extracts them in config.data_root (~10-15 min)
!python scripts/download_dataset.py --config configs/config.yaml

In [ ]:
# 1.5 Weights & Biases login (API key from Colab Secrets: WANDB_API_KEY)
import wandb
from google.colab import userdata

try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    wandb.login()
    print(f"W&B ready: project={config.wandb.project}, entity={config.wandb.entity}")
except Exception as e:
    print(f"W&B not configured ({e}).")
    print("Add the WANDB_API_KEY secret, or set wandb.mode: 'disabled' in configs/config.yaml")

In [ ]:
# 1.6 Imports and environment check
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data import DexYCBDataset, get_transforms
from src.utils.checkpoint import load_checkpoint
from src.utils.metrics import compute_mpjpe

device = torch.device(config.device if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | device: {device}")
print(f"Data root: {config.data_root} | batch size: {config.data.batch_size}")

print("\nSETUP COMPLETE")

---
## 2. DATA EXPLORATION - Inspect DexYCB (Part 1)

Layout of `config.data_root` after 1.4:
```
20200709-subject-01/<sequence>/meta.yml                      # objects (ycb_ids), grasped object (ycb_grasp_ind), hand side
20200709-subject-01/<sequence>/<camera serial>/color_XXXXXX.jpg
                                              aligned_depth_to_color_XXXXXX.png
                                              labels_XXXXXX.npz  # seg, pose_y (objects), pose_m (MANO), joint_3d, joint_2d
calibration/   # camera intrinsics/extrinsics, MANO betas per subject
models/        # YCB object meshes
```

In [ ]:
# 2.1 Raw look at one frame (no Dataset class needed)
import glob

import matplotlib.pyplot as plt
import yaml

subject_dir = f"{config.data_root}/{config.dexycb.subject}"
seq = sorted(os.listdir(subject_dir))[0]
meta = yaml.safe_load(open(f"{subject_dir}/{seq}/meta.yml"))
cam = meta["serials"][0]
print(f"sequence {seq} | {len(os.listdir(subject_dir))} sequences | {meta['num_frames']} frames | 8 cameras")
print(f"objects (ycb_ids): {meta['ycb_ids']} | grasped: {meta['ycb_ids'][meta['ycb_grasp_ind']]} | hand: {meta['mano_sides']}")

frame = 40
labels = np.load(f"{subject_dir}/{seq}/{cam}/labels_{frame:06d}.npz")
for k in labels.files:
    print(f"  {k:10s} {labels[k].shape} {labels[k].dtype}")

img = plt.imread(f"{subject_dir}/{seq}/{cam}/color_{frame:06d}.jpg")
j2d = labels["joint_2d"].reshape(-1, 2)
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.scatter(j2d[:, 0], j2d[:, 1], c="lime", s=15)
plt.title(f"{seq} / camera {cam} / frame {frame}")
plt.axis("off")
plt.show()

In [ ]:
# 2.2 Dataset + DataLoader  (requires DexYCBDataset in src/data/dataset.py)
transform = get_transforms("train", config)
train_dataset = DexYCBDataset("train", config, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=config.data.batch_size,
                          num_workers=config.data.num_workers, shuffle=True)

print(f"Train dataset size: {len(train_dataset)}")

In [ ]:
# 2.3 Inspect a batch
batch = next(iter(train_loader))

print("Batch keys:", batch.keys())
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(f"  {k}: shape={tuple(v.shape)}, dtype={v.dtype}")
    else:
        print(f"  {k}: {type(v)}")

In [ ]:
# 2.4 Visualize RGB images from the batch (undo the ImageNet normalization)
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

rgb_batch = batch["rgb"]  # (B, 3, H, W)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(min(8, rgb_batch.shape[0])):
    ax = axes[i // 4, i % 4]
    img = (rgb_batch[i] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"Sample {i}")
plt.tight_layout()
plt.show()

---
## 3. HaMeR PRETRAINED INFERENCE - Table 1

TODO: `src/models/hamer` is a placeholder (timm ViT-Base): it must be replaced by the official
pretrained HaMeR (https://github.com/geopavlakos/hamer, ViT-H backbone, needs MANO).

In [ ]:
# 3.1 Load pretrained HaMeR
from src.models.hamer import HaMeR

model = HaMeR.load_from_checkpoint(config.hamer.pretrained_path).to(device)
model.eval()

print(f"HaMeR pretrained loaded on {device}")

In [ ]:
# 3.2 Validation dataloader
val_transform = get_transforms("val", config)
val_dataset = DexYCBDataset("val", config, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=config.data.batch_size,
                        num_workers=config.data.num_workers)

print(f"Val dataset size: {len(val_dataset)}")

In [ ]:
# 3.3 Evaluation loop (vertex metrics require the MANO layer)
all_mpjpe = []

with torch.no_grad():
    for batch in val_loader:
        rgb = batch["rgb"].to(device)
        pred = model(rgb)
        if "joints_3d" in batch:
            all_mpjpe.append(compute_mpjpe(pred["joints_3d"], batch["joints_3d"].to(device)))

if all_mpjpe:
    print(f"MPJPE: {np.mean(all_mpjpe):.2f} mm")
print("TODO: full Table 1 metrics with the MANO layer (smplx)")

---
## 4. HaMeR FINE-TUNING on DexYCB - Table 1

In [ ]:
# 4.1 Model, optimizer, dataloaders
from src.models.hamer import HaMeR
from src.training import Trainer

hamer_cfg = config.train.hamer
model = HaMeR(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=hamer_cfg.lr, weight_decay=hamer_cfg.weight_decay)
# loss_fn = ...  (TODO: HaMeR loss in src/models/losses.py)

train_dataset = DexYCBDataset("train", config, transform=get_transforms("train", config))
val_dataset = DexYCBDataset("val", config, transform=get_transforms("val", config))
train_loader = DataLoader(train_dataset, batch_size=config.data.batch_size,
                          num_workers=config.data.num_workers, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.data.batch_size,
                        num_workers=config.data.num_workers)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# 4.2 Training (TODO: Trainer.fit in src/training/trainer.py)
# trainer = Trainer(model, train_loader, val_loader, optimizer, loss_fn, config, device,
#                   checkpoint_dir=f"{config.checkpoint_dir}/{hamer_cfg.checkpoint_subdir}",
#                   train_cfg=hamer_cfg)
# trainer.fit(hamer_cfg.epochs, run_name="hamer-finetuned")

print(f"Epochs: {hamer_cfg.epochs}, LR: {hamer_cfg.lr}")

---
## 5. HOI MODEL TRAINING - Part 3

In [ ]:
# 5.1 Model, optimizer, loss, dataloaders
from src.models.hoi import HOIModel
from src.models.losses import HOILoss

hoi_cfg = config.train.hoi
model = HOIModel(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=hoi_cfg.lr, weight_decay=hoi_cfg.weight_decay)
loss_fn = HOILoss(config)

train_dataset = DexYCBDataset("train", config, transform=get_transforms("train", config))
val_dataset = DexYCBDataset("val", config, transform=get_transforms("val", config))
train_loader = DataLoader(train_dataset, batch_size=config.data.batch_size,
                          num_workers=config.data.num_workers, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.data.batch_size,
                        num_workers=config.data.num_workers)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# 5.2 Training: checkpoints in <your Drive>/HOI_Project/checkpoints/hoi_model, metrics on W&B
run = wandb.init(project=config.wandb.project, entity=config.wandb.entity, mode=config.wandb.mode,
                 name="hoi-training", config=dict(hoi_cfg))

trainer = Trainer(model, train_loader, val_loader, optimizer, loss_fn, config, device,
                  checkpoint_dir=f"{config.checkpoint_dir}/{hoi_cfg.checkpoint_subdir}",
                  train_cfg=hoi_cfg)
trainer.fit(hoi_cfg.epochs, run_name="hoi-training")

run.finish()
print("HOI training complete")

---
## 6. EVALUATION - Table 2 metrics + Visualizations (Figure 4)

In [ ]:
# 6.1 Load the trained HOI model
model = HOIModel(config).to(device)
checkpoint_path = f"{config.checkpoint_dir}/{config.train.hoi.checkpoint_subdir}/best.pth"
epoch, metrics = load_checkpoint(checkpoint_path, model, map_location=device)
model.eval()

print(f"Loaded checkpoint from epoch {epoch}")

In [ ]:
# 6.2 Test dataloader
test_dataset = DexYCBDataset("test", config, transform=get_transforms("test", config))
test_loader = DataLoader(test_dataset, batch_size=config.data.batch_size,
                         num_workers=config.data.num_workers)

print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
# 6.3 Evaluation loop - Table 2 metrics
all_metrics = []

with torch.no_grad():
    for batch in test_loader:
        rgb = batch["rgb"].to(device)
        obj_class = batch["obj_class"].to(device)
        pred = model(rgb, obj_class)
        # metrics = compute_all_metrics(pred, batch, mano_layer, obj_mesh)
        # all_metrics.append(metrics)

print("TODO: full Table 2 metrics (MPJPE, PA, RR, AUC, MPVPE, ADD, PD) with the MANO layer")

In [ ]:
# 6.4 Qualitative visualizations (Figure 4) -> saved in config.output_dir
# for batch in test_loader:
#     pred = model(batch["rgb"].to(device), batch["obj_class"].to(device))
#     vis = overlay_predictions(...)

print("TODO: qualitative visualizations (pyrender, trimesh, MANO layer, YCB meshes in config.data_root/models)")

---
## Before closing Colab
- Code changes made here are lost when the session ends: commit and push them from your PC.
- Checkpoints and outputs are already on your Drive (`MyDrive/HOI_Project`).